# 05 - Real Memory Lifecycle

## Objetivo
Observar a máquina de estados real de `MemoryInfo`
(`RAW -> OCCUPIED -> ENTANGLED -> (PURIFIED | RAW) -> ...`) memória a
memória, com timestamp e o protocolo/mecanismo responsável por cada
transição - sem forçar nenhum estado manualmente.

## Conceitos
- `MemoryInfo.to_raw/to_occupied/to_entangled/to_purified`
  (`sequence/resource_management/memory_manager.py`) são os **únicos**
  lugares onde `MemoryInfo.state` é atribuído, mas nenhum deles recebe um
  argumento dizendo quem chamou. `ibqn.demos.instrumentation.MemoryLifecycleRecorder`
  identifica o responsável lendo o grafo de chamadas real: monkeypatcha
  `ResourceManager.update`/`.load` (que sabem qual `protocol`/`Rule` está
  agindo) só para manter um rótulo "responsável atual", e monkeypatcha os
  4 métodos `to_*` de cada `MemoryInfo` para gravar a transição completa -
  tudo em métodos de instâncias específicas, nenhuma classe do SeQUeNCe é
  alterada.
- Um wrap ingênuo (só em `ResourceManager.update`) esconderia transições
  intermediárias que acontecem dentro da mesma chamada (ex.: `to_occupied()`
  chamado diretamente por `ResourceManager.load`/`.update` quando uma regra
  recém-instalada já casa imediatamente) - por isso o recorder também
  intercepta os 4 métodos `to_*` diretamente, garantindo a sequência
  completa e não apenas o resultado final.
- Este notebook usa o mesmo cenário com purificação do notebook anterior
  (`03_sequence_purification.ipynb`, Case B) especificamente para exercitar
  os 4 estados, incluindo tentativas de purificação malsucedidas (memória
  volta para `RAW`).


## Configuração


In [1]:
from ibqn.demos.topologies import three_node_spec
from ibqn.demos.instrumentation import MemoryLifecycleRecorder
from ibqn.demos.tables import memory_transitions_table
from ibqn.network.sequence_adapter import SequenceAdapter
from ibqn.network.capabilities import NetworkCapabilities
from ibqn.execution.sequence_executor import SequenceExecutor
from ibqn.intent.repository import IntentRepository
from ibqn.intent.models import (
    EntanglementIntent, IntentEndpoints, IntentRequirements, IntentPolicy, IntentValidation, SuccessCondition,
)
from ibqn.planning.planner import IntentPlanner

SEED = 0
REQUESTED_PAIRS = 10
MIN_FIDELITY = 0.75  # above the swap-only fidelity (0.73): forces purification, see notebook 03
RAW_FIDELITY = 0.85

spec = three_node_spec(
    end_memories=REQUESTED_PAIRS, repeater_memories=REQUESTED_PAIRS * 2,
    raw_fidelity=RAW_FIDELITY,
    stop_time_s=0.12,
)

intent = EntanglementIntent(
    id="intent-memory-lifecycle-demo",
    endpoints=IntentEndpoints(source="a", destination="b"),
    requirements=IntentRequirements(
        min_fidelity=MIN_FIDELITY, min_throughput=1, max_latency=1.0,
        requested_pairs=REQUESTED_PAIRS, start_time=0.02, duration=0.06,
    ),
    policy=IntentPolicy(allow_purification=True),
    validation=IntentValidation(
        metrics=["delivered_pairs"],
        success_conditions=[SuccessCondition(metric="delivered_pairs", operator=">=", expected=1)],
    ),
)

capabilities = NetworkCapabilities(spec)
planner = IntentPlanner(capabilities)
plan = planner.plan(intent)
print(f"route: {' -> '.join(plan.route)}, requires_purification: {plan.requires_purification}")


2026-10-02 15:03:24,648 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'requested_pairs' -> 'reserved_memory_slots' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:24,654 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'start_time' -> 'start_time_s' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:24,655 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'duration' -> 'duration_s' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:24,660 INFO     ibqn.ibqn.planning.planner intent_id=intent-memory-lifecycle-demo - planned route a->r->b, estimated_fidelity=0.7676, purification=True


route: a -> r -> b, requires_purification: True


## Execução

`MemoryLifecycleRecorder` é atrelado a cada roteador **antes** de
`executor.run()` - ele só observa; nenhum estado é escrito por este
notebook.


In [2]:
adapter = SequenceAdapter(spec, seed=SEED)
repository = IntentRepository()
executor = SequenceExecutor(adapter, repository)
executor.deploy(intent, plan)

recorders = {
    node_id: MemoryLifecycleRecorder(adapter.get_router(node_id))
    for node_id in ("a", "r", "b")
}

executor.run()

repository.get(intent.id).lifecycle.status


2026-10-02 15:03:24,688 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - topology built: 3 routers, 2 quantum links, formalism=bell_diagonal, seed=0


2026-10-02 15:03:24,691 INFO     ibqn.ibqn.intent.repository intent_id=intent-memory-lifecycle-demo - intent registered


2026-10-02 15:03:24,693 INFO     ibqn.ibqn.intent.repository intent_id=intent-memory-lifecycle-demo - intent transitioned to VALIDATED (schema validated)


2026-10-02 15:03:24,694 INFO     ibqn.ibqn.intent.repository intent_id=intent-memory-lifecycle-demo - intent transitioned to PLANNING (invoking IntentPlanner)


2026-10-02 15:03:24,696 INFO     ibqn.ibqn.intent.repository intent_id=intent-memory-lifecycle-demo - intent transitioned to PLANNED (selected by ShortestHopCountRouting; hop fidelities [0.85, 0.85])


2026-10-02 15:03:24,697 INFO     ibqn.ibqn.intent.repository intent_id=intent-memory-lifecycle-demo - intent transitioned to DEPLOYING (applying route a->r->b and submitting reservation to NetworkManager (purification_mode=until_target))


2026-10-02 15:03:24,698 INFO     ibqn.ibqn.execution.sequence_executor intent_id=intent-memory-lifecycle-demo - submitting reservation a -> b, reserved_memory_slots=10, min_fidelity=0.750, purification_mode=until_target


2026-10-02 15:03:24,701 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - running simulation, stop_time=0.120s


2026-10-02 15:03:24,705 INFO     ibqn.ibqn.intent.repository intent_id=intent-memory-lifecycle-demo - intent transitioned to ACTIVE (reservation approved)


<IntentStatus.ACTIVE: 'ACTIVE'>

## Resultados


In [3]:
import pandas as pd

per_node_tables = []
for node_id, recorder in recorders.items():
    node_df = memory_transitions_table(recorder)
    node_df.insert(1, "node", node_id)
    per_node_tables.append(node_df)

transitions_df = pd.concat(per_node_tables, ignore_index=True).sort_values("sim_time_s").reset_index(drop=True)
print(f"total recorded transitions across all 3 nodes: {len(transitions_df)}")
transitions_df.head(15)


total recorded transitions across all 3 nodes: 12100


,sim_time_s,node,memory,from_state,to_state,responsible
0,0.02,a,a.MemoryArray[0],RAW,OCCUPIED,Rule installation (priority=10)
1,0.02,a,a.MemoryArray[1],RAW,OCCUPIED,Rule installation (priority=10)
2,0.02,a,a.MemoryArray[2],RAW,OCCUPIED,Rule installation (priority=10)
3,0.02,r,r.MemoryArray[18],RAW,OCCUPIED,Rule installation (priority=10)
4,0.02,r,r.MemoryArray[19],RAW,OCCUPIED,Rule installation (priority=10)
5,0.02,r,r.MemoryArray[16],RAW,OCCUPIED,Rule installation (priority=10)
6,0.02,r,r.MemoryArray[2],RAW,OCCUPIED,Rule installation (priority=10)
7,0.02,r,r.MemoryArray[1],RAW,OCCUPIED,Rule installation (priority=10)
8,0.02,r,r.MemoryArray[15],RAW,OCCUPIED,Rule installation (priority=10)
9,0.02,r,r.MemoryArray[14],RAW,OCCUPIED,Rule installation (priority=10)


In [4]:
observed_states = sorted(transitions_df["to_state"].unique())
print(f"states observed: {observed_states}")
assert set(observed_states) >= {"RAW", "OCCUPIED", "ENTANGLED", "PURIFIED"}, (
    "expected all 4 real MemoryInfo states to appear at least once in this purification-forcing scenario"
)


states observed: ['ENTANGLED', 'OCCUPIED', 'PURIFIED', 'RAW']


In [5]:
aggregate_df = (
    transitions_df
    .groupby(["node", "from_state", "to_state", "responsible"])
    .size()
    .reset_index(name="occurrences")
    .sort_values("occurrences", ascending=False)
)
aggregate_df.head(20)


,node,from_state,to_state,responsible,occurrences
34,r,OCCUPIED,RAW,SingleHeraldedA,1244
38,r,RAW,OCCUPIED,SingleHeraldedA,1227
7,a,OCCUPIED,RAW,SingleHeraldedA,619
22,b,OCCUPIED,RAW,SingleHeraldedA,617
12,a,RAW,OCCUPIED,SingleHeraldedA,614
27,b,RAW,OCCUPIED,SingleHeraldedA,613
31,r,OCCUPIED,ENTANGLED,SingleHeraldedA,603
33,r,OCCUPIED,RAW,EntanglementSwappingA_BDS,602
29,r,ENTANGLED,OCCUPIED,SingleHeraldedA,602
36,r,RAW,OCCUPIED,EntanglementSwappingA_BDS,600


In [6]:
responsible_labels = set(transitions_df["responsible"])
print("distinct 'responsible' labels observed:")
for label in sorted(responsible_labels):
    print(f"  - {label}")
assert any("Rule installation" in label for label in responsible_labels), (
    "expected at least one transition attributed to a freshly-installed Rule"
)
assert any(label not in ("(unattributed)",) and "Rule installation" not in label and "no protocol" not in label
           for label in responsible_labels), (
    "expected at least one transition attributed to a concrete entanglement protocol class"
)


distinct 'responsible' labels observed:
  - (no protocol - explicit reset, e.g. by an application or a reservation's end-time cleanup)
  - BBPSSW_BDS
  - EntanglementSwappingA_BDS
  - EntanglementSwappingB_BDS
  - Rule installation (priority=10)
  - SingleHeraldedA


## Interpretação
A tabela agregada mostra que memórias passam por `OCCUPIED` assim que uma
`Rule` é instalada (responsável: "Rule installation"), depois por
`ENTANGLED` quando um protocolo de geração/swap tem sucesso (responsável:
o nome real da classe do protocolo, ex. `SingleHeraldedA`,
`EntanglementSwappingB_BDS`), e finalmente por `PURIFIED` (quando um
round de purificação bem-sucedido ocorre) ou de volta a `RAW` (reset
explícito por `IntentRequestApp.get_memory`/`_count_purified_delivery`,
ou uma tentativa de purificação malsucedida) - nenhuma dessas transições
foi escrita por este notebook; todas vêm do mecanismo real de regras do
SeQUeNCe (`ResourceManager`/`RuleManager`) reagindo a estados de memória
observados durante a simulação.

## Limitações
- `MemoryLifecycleRecorder` não distingue duas tentativas de purificação
  que ocorrem com o mesmo par de memórias no mesmo timestamp (resolução de
  tempo é a do `Timeline`, em picossegundos, convertida para segundos).
- Um cenário sem purificação (`min_fidelity` mais baixo) não exercitaria o
  estado `PURIFIED` - por isso este notebook usa deliberadamente o mesmo
  requisito do Case B em `03_sequence_purification.ipynb`.
- O volume de transições cresce rapidamente com `stop_time_s`; este
  notebook usa uma janela deliberadamente curta para manter a tabela
  agregada legível.

## Conclusão
Máquina de estados real de memória demonstrada com atribuição correta de
responsabilidade por transição, obtida inteiramente por composição
(monkeypatching de métodos de instância), sem qualquer modificação do
código-fonte do SeQUeNCe.
